In [1]:
%load_ext autoreload
%autoreload 2

In [8]:
import numpy as np
import torch
from bridgit import Bridgit, Player
from bridgit.schema import Move
from bridgit.ai import BridgitNet, NetWrapper, MCTS
from bridgit.config import BoardConfig, MCTSConfig, NeuralNetConfig
from bridgit import schema
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [9]:
board_config = BoardConfig(size=5)
mcts_config = MCTSConfig(num_simulations=5000)
net_config = NeuralNetConfig(num_channels=32, num_res_blocks=2)

net = BridgitNet(board_config, net_config)
wrapper = NetWrapper(net)
mcts = MCTS(wrapper, mcts_config, board_config)

game = Bridgit(board_config)

g = board_config.grid_size
print(f"Board: {g}x{g}")
print(f"Legal crossings: {len(game.get_available_moves())}")
print(f"Device: {wrapper.device}")

Board: 11x11
Legal crossings: 41
Device: mps


In [10]:
game = Bridgit(board_config)
game.make_move(schema.Move(row=1, col=1))
game.make_move(schema.Move(row=2, col=2))
game.make_move(schema.Move(row=3, col=3))
g = board_config.grid_size

# Get raw net prediction
policy, value = mcts._predict(game)

# Mask to only legal crossings
mask = game.to_mask()
masked_policy = policy * mask
masked_policy /= masked_policy.sum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Raw Policy (all cells)", "Masked Policy (legal moves only)"])

fig.add_trace(go.Heatmap(z=np.flipud(policy), colorscale="Blues", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=np.flipud(masked_policy), colorscale="Reds", showscale=False), row=1, col=2)

fig.update_layout(width=700, height=350, title=f"Net value estimate: {value:.3f}")
fig.show()

In [20]:
game = Bridgit(board_config)
game.make_move(schema.Move(row=1, col=1))
game.make_move(schema.Move(row=2, col=2))
game.make_move(schema.Move(row=3, col=3))

# MCTS search
visit_counts = mcts._search(game).visit_counts()
mcts_probs = mcts.get_action_probs(game, temperature=1.0)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Visit Counts", "MCTS Policy (temp=1)"])

fig.add_trace(go.Heatmap(z=np.flipud(visit_counts), colorscale="Greens", showscale=True), row=1, col=1)
fig.add_trace(go.Heatmap(z=np.flipud(mcts_probs), colorscale="Reds", showscale=True), row=1, col=2)

fig.update_layout(width=700, height=350, title="MCTS concentrates visits on promising moves")
fig.show()

# # Print top moves
# print(f"Player: {game.current_player.name}, moves_left: {game.moves_left_in_turn}")
# print(f"\nTop 5 moves by visit count:")
# legal_actions = np.nonzero(gw.get_valid_moves_mask(game))[0]
# sorted_actions = sorted(legal_actions, key=lambda a: visit_counts[a], reverse=True)
# for a in sorted_actions[:5]:
#     r, c = gw.action_to_move(a)
#     print(f"  ({r},{c})  visits={int(visit_counts[a]):3d}  prob={mcts_probs[a]:.3f}")

In [18]:
mcts_probs

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0246, 0.0000, 0.0198, 0.0000, 0.0212, 0.0000,
         0.0322, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0228, 0.0000, 0.0212, 0.0000, 0.0208,
         0.0000, 0.0000],
        [0.0000, 0.0214, 0.0000, 0.0000, 0.0000, 0.0198, 0.0000, 0.0224, 0.0000,
         0.0280, 0.0000],
        [0.0000, 0.0000, 0.0508, 0.0000, 0.0318, 0.0000, 0.0200, 0.0000, 0.0196,
         0.0000, 0.0000],
        [0.0000, 0.0320, 0.0000, 0.0224, 0.0000, 0.0362, 0.0000, 0.0206, 0.0000,
         0.0234, 0.0000],
        [0.0000, 0.0000, 0.0238, 0.0000, 0.0228, 0.0000, 0.0288, 0.0000, 0.0336,
         0.0000, 0.0000],
        [0.0000, 0.0410, 0.0000, 0.0270, 0.0000, 0.0312, 0.0000, 0.0284, 0.0000,
         0.0212, 0.0000],
        [0.0000, 0.0000, 0.0212, 0.0000, 0.0384, 0.0000, 0.0324, 0.0000, 0.0172,
         0.0000, 0.0000],
        [0.0000, 0.0292, 0.0000, 0.02